# Étape 4 – Data Lake sur AWS S3

**Objectif** : stocker toutes les données collectées dans un **Data Lake** (bucket S3), dont le fichier demandé
par l'énoncé : *un `.csv` contenant les informations enrichies sur la météo et les hôtels de chaque ville*.

**Compétence du référentiel** : C1 – concevoir une architecture de données (**Data Lake**) répondant aux besoins
de stockage, d'utilisation, de **sécurité** et de **protection**.

**Organisation du Data Lake** :

| Zone | Contenu | Règle |
|---|---|---|
| `raw/` | Données **brutes**, telles que collectées | Jamais modifiées ; rangées par **date de collecte** (`collected_date=AAAA-MM-JJ`) pour garder l'historique |
| `processed/` | Données **nettoyées et enrichies** | Dernière version, remplacée à chaque exécution ; source de l'ETL vers RDS (étape 5) |

**Sécurité** :

| Mesure | Mise en œuvre |
|---|---|
| Bucket privé | *Block all public access* activé |
| Moindre privilège | L'utilisateur `kayak-project` peut seulement **lister, lire, écrire, supprimer** des fichiers dans **ce** bucket ; il ne peut ni créer, ni supprimer, ni configurer de bucket (créé dans la console par l'administrateur) |
| Chiffrement | SSE-S3 (chiffrement côté serveur, par défaut) |
| Données en France | Région `eu-west-3` (Paris), cohérent avec le RGPD |
| Récupération | Versioning activé : un fichier écrasé ou supprimé par erreur reste récupérable |
| Secrets | Clés lues depuis `.env` (exclu de Git) ; boto3 les lit automatiquement dans les variables d'environnement |

## 1. Imports, configuration et connexion à AWS

In [1]:
import os
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path

import boto3
import pandas as pd
from botocore.exceptions import ClientError
from dotenv import load_dotenv


def find_project_root(start: Path) -> Path:
    """Remonte l'arborescence jusqu'au dossier qui contient requirements.txt (racine du projet)."""
    for folder in [start, *start.parents]:
        if (folder / "requirements.txt").exists():
            return folder
    raise FileNotFoundError("Racine du projet introuvable (requirements.txt absent)")


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

load_dotenv(PROJECT_ROOT / ".env")  # AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_DEFAULT_REGION, S3_BUCKET_NAME
BUCKET = os.getenv("S3_BUCKET_NAME")
REGION = os.getenv("AWS_DEFAULT_REGION")

s3 = boto3.client("s3")  # les identifiants sont lus dans les variables d'environnement : aucun secret dans le code
identity = boto3.client("sts").get_caller_identity()
print("Identité AWS utilisée :", identity["Arn"].split(":")[-1])
print("Bucket cible          :", BUCKET, "| région :", REGION)

Identité AWS utilisée : user/kayak-project
Bucket cible          : kayak-plan-your-trip-fo | région : eu-west-3


## 2. Construction du fichier enrichi (livrable de l'énoncé)

Une ligne par **hôtel et par destination**, complétée par la **météo** de la destination (indicateurs à 7 jours,
score et rang). On distingue clairement les coordonnées de la destination (`city_latitude`, `city_longitude`)
de celles de l'hôtel (`hotel_latitude`, `hotel_longitude`).

In [2]:
cities = pd.read_csv(RAW_DIR / "cities.csv")
weather_summary = pd.read_csv(PROCESSED_DIR / "weather_summary.csv")
hotels = pd.read_csv(PROCESSED_DIR / "hotels.csv")

destinations = (
    weather_summary
    .rename(columns={"latitude": "city_latitude", "longitude": "city_longitude",
                     "period_start": "weather_period_start", "period_end": "weather_period_end"})
)
kayak_enriched = (
    hotels
    .rename(columns={"latitude": "hotel_latitude", "longitude": "hotel_longitude",
                     "collected_at": "hotels_collected_at"})
    .merge(destinations, on="city_id", how="left", validate="many_to_one")
)

# Ordre des colonnes : destination, météo, puis hôtel
ENRICHED_COLUMNS = [
    "city_id", "city_name", "city_latitude", "city_longitude",
    "weather_period_start", "weather_period_end", "total_rain_mm", "avg_pop", "avg_temp_day", "rainy_days",
    "weather_score", "weather_rank",
    "hotel_id", "hotel_name", "hotel_url", "search_position", "hotel_latitude", "hotel_longitude", "address",
    "hotel_city", "distance_km", "review_score", "review_count", "star_rating", "description", "hotels_collected_at",
]
kayak_enriched = kayak_enriched[ENRICHED_COLUMNS].sort_values(["weather_rank", "city_id", "search_position"])

assert len(kayak_enriched) == len(hotels), "La jointure ne doit ni perdre ni dupliquer de lignes"
assert kayak_enriched["weather_score"].notna().all(), "Chaque hôtel doit avoir la météo de sa destination"

ENRICHED_FILE = PROCESSED_DIR / "kayak_enriched.csv"
kayak_enriched.to_csv(ENRICHED_FILE, index=False, encoding="utf-8")
print(f"{len(kayak_enriched)} lignes × {kayak_enriched.shape[1]} colonnes -> {ENRICHED_FILE.name}")
kayak_enriched.head(3)

700 lignes × 26 colonnes -> kayak_enriched.csv


,city_id,city_name,city_latitude,city_longitude,weather_period_start,weather_period_end,total_rain_mm,avg_pop,avg_temp_day,rainy_days,...,hotel_latitude,hotel_longitude,address,hotel_city,distance_km,review_score,review_count,star_rating,description,hotels_collected_at
540,28,Collioure,42.52505,3.083155,2026-11-05,2026-11-11,1.0,0.27,16.8,3,...,42.525348,3.090324,Route de Port-Vendres,Collioure,0.6,9.3,653,4.0,L’établissement Les Roches Brunes se trouve à ...,2026-11-05T09:05:26+00:00
541,28,Collioure,42.52505,3.083155,2026-11-05,2026-11-11,1.0,0.27,16.8,3,...,42.526167,3.082356,Impasse Des Palmiers,Collioure,0.1,9.2,1007,4.0,Situé à 250 mètres de la plage et du port de C...,2026-11-05T09:05:26+00:00
542,28,Collioure,42.52505,3.083155,2026-11-05,2026-11-11,1.0,0.27,16.8,3,...,42.528883,3.079394,33 Route du Pla de las Fourques,Collioure,0.5,9.1,665,5.0,L’établissement Maison Nova - Boutique Hôtel &...,2026-11-05T09:05:26+00:00


## 3. Plan de chargement : quel fichier, vers quelle clé S3 ?

La **date de collecte** de chaque jeu de données brut est lue dans les données elles-mêmes
(colonne `collected_at`) ; à défaut (`cities.csv`), on utilise la date du fichier.

In [3]:
def collected_date_from_column(path: Path, column: str = "collected_at") -> str:
    """Date (AAAA-MM-JJ) de collecte d'un CSV, lue dans sa colonne de date de collecte."""
    return str(pd.read_csv(path, usecols=[column])[column].iloc[0])[:10]


def file_date(path: Path) -> str:
    """Date (AAAA-MM-JJ) de dernière modification d'un fichier."""
    return datetime.fromtimestamp(path.stat().st_mtime, tz=timezone.utc).strftime("%Y-%m-%d")


weather_date = collected_date_from_column(RAW_DIR / "weather_daily.csv")
hotels_date = collected_date_from_column(PROCESSED_DIR / "hotels.csv")

UPLOAD_PLAN = [
    # (fichier local, clé S3)
    (RAW_DIR / "cities.csv",            f"raw/cities/collected_date={file_date(RAW_DIR / 'cities.csv')}/cities.csv"),
    (RAW_DIR / "weather_daily.csv",     f"raw/weather/collected_date={weather_date}/weather_daily.csv"),
    (RAW_DIR / "hotels_booking.json",   f"raw/hotels/collected_date={hotels_date}/hotels_booking.json"),
    (PROCESSED_DIR / "weather_summary.csv", "processed/weather_summary.csv"),
    (PROCESSED_DIR / "hotels.csv",          "processed/hotels.csv"),
    (PROCESSED_DIR / "kayak_enriched.csv",  "processed/kayak_enriched.csv"),
]

for local_path, key in UPLOAD_PLAN:
    assert local_path.exists(), f"Fichier manquant : {local_path}"
    print(f"{local_path.name:<22} -> s3://{BUCKET}/{key}")

cities.csv             -> s3://kayak-plan-your-trip-fo/raw/cities/collected_date=2026-11-04/cities.csv
weather_daily.csv      -> s3://kayak-plan-your-trip-fo/raw/weather/collected_date=2026-11-04/weather_daily.csv
hotels_booking.json    -> s3://kayak-plan-your-trip-fo/raw/hotels/collected_date=2026-11-05/hotels_booking.json
weather_summary.csv    -> s3://kayak-plan-your-trip-fo/processed/weather_summary.csv
hotels.csv             -> s3://kayak-plan-your-trip-fo/processed/hotels.csv
kayak_enriched.csv     -> s3://kayak-plan-your-trip-fo/processed/kayak_enriched.csv


## 4. Envoi des fichiers dans S3

In [4]:
CONTENT_TYPES = {".csv": "text/csv; charset=utf-8", ".json": "application/json"}

for local_path, key in UPLOAD_PLAN:
    s3.upload_file(
        Filename=str(local_path),
        Bucket=BUCKET,
        Key=key,
        ExtraArgs={"ContentType": CONTENT_TYPES[local_path.suffix]},
    )
    print(f"✅ {key}")

✅ raw/cities/collected_date=2026-11-04/cities.csv
✅ raw/weather/collected_date=2026-11-04/weather_daily.csv
✅ raw/hotels/collected_date=2026-11-05/hotels_booking.json
✅ processed/weather_summary.csv
✅ processed/hotels.csv
✅ processed/kayak_enriched.csv


## 5. Vérifications

### 5.1 Contenu du bucket

In [5]:
listing = s3.list_objects_v2(Bucket=BUCKET)
objects = pd.DataFrame(
    [{"key": o["Key"], "size_kb": round(o["Size"] / 1024, 1), "last_modified": o["LastModified"]} for o in listing.get("Contents", [])]
)
objects

,key,size_kb,last_modified
0,processed/hotels.csv,264.5,2026-11-05 15:12:58+00:00
1,processed/kayak_enriched.csv,316.6,2026-11-05 15:12:58+00:00
2,processed/weather_summary.csv,2.9,2026-11-05 15:12:58+00:00
3,raw/cities/collected_date=2026-11-04/cities.csv,5.3,2026-11-05 15:12:57+00:00
4,raw/hotels/collected_date=2026-11-05/hotels_bo...,447.6,2026-11-05 15:12:57+00:00
5,raw/weather/collected_date=2026-11-04/weather_...,38.9,2026-11-05 15:12:57+00:00


### 5.2 Relecture depuis S3 et chiffrement

On relit le fichier enrichi **depuis S3** (comme le fera l'ETL) et on vérifie qu'il est identique au fichier local,
puis on contrôle le chiffrement côté serveur.

In [6]:
ENRICHED_KEY = "processed/kayak_enriched.csv"
remote = pd.read_csv(BytesIO(s3.get_object(Bucket=BUCKET, Key=ENRICHED_KEY)["Body"].read()))
print("Fichier relu depuis S3 :", remote.shape, "| identique au fichier local :", remote.shape == kayak_enriched.shape)

head = s3.head_object(Bucket=BUCKET, Key=ENRICHED_KEY)
print("Chiffrement côté serveur :", head.get("ServerSideEncryption"))   # AES256 = SSE-S3
print("Version de l'objet       :", head.get("VersionId"))              # présent si le versioning est activé

Fichier relu depuis S3 : (700, 26) | identique au fichier local : True
Chiffrement côté serveur : AES256
Version de l'objet       : fOsINRuM5tcMOMdxjP7G21xX_mVGcDyc


### 5.3 Démonstration du moindre privilège

L'utilisateur `kayak-project` ne doit **pas** pouvoir lister les autres buckets du compte ni modifier
la configuration du bucket. Une erreur `AccessDenied` est ici le **résultat attendu**.

In [7]:
def try_action(label: str, action) -> None:
    """Exécute une action AWS et affiche si elle est autorisée ou refusée."""
    try:
        action()
        print(f"⚠️ {label} : AUTORISÉ (droits trop larges ?)")
    except ClientError as error:
        print(f"✅ {label} : refusé ({error.response['Error']['Code']})")


try_action("Lister tous les buckets du compte", lambda: s3.list_buckets())
try_action("Lire la configuration d'accès public", lambda: s3.get_public_access_block(Bucket=BUCKET))
try_action("Créer un autre bucket", lambda: s3.create_bucket(
    Bucket=f"{BUCKET}-test", CreateBucketConfiguration={"LocationConstraint": REGION}))

✅ Lister tous les buckets du compte : refusé (AccessDenied)
✅ Lire la configuration d'accès public : refusé (AccessDenied)
✅ Créer un autre bucket : refusé (AccessDenied)
